In [3]:
import os
import glob
import numpy as np
import pandas as pd
import dill as pickle
from loguru import logger

def find_all_sessions(base_path="Z:/Jasmine_Laurence/Experimental_Data"):
    session_paths = []
    mouse_dirs = glob.glob(os.path.join(base_path, "JAL*"))
    for mouse_dir in mouse_dirs:
        sessions = glob.glob(os.path.join(mouse_dir, "*"))
        for session in sessions:
            if os.path.isdir(session) and os.path.exists(os.path.join(session, "processed_data")):
                session_paths.append(session)
    return session_paths

def load_homings_object(session_path):
    homings_path = os.path.join(session_path, "processed_data", "homings", "homings_obj.pkl")
    if os.path.exists(homings_path):
        try:
            with open(homings_path, "rb") as f:
                return pickle.load(f)
        except Exception as e:
            logger.warning(f"Error loading homings object from {session_path}: {e}")
    return None

def load_escapes_object(session_path):
    possible_paths = [
        os.path.join(session_path, "processed_data", "escapes", "escapes_obj.pkl"),
        os.path.join(session_path, "processed_data", "escape", "escapes_obj.pkl"),
        os.path.join(session_path, "processed_data", "escapes_obj.pkl")
    ]
    for path in possible_paths:
        if os.path.exists(path):
            try:
                with open(path, "rb") as f:
                    return pickle.load(f)
            except Exception as e:
                logger.warning(f"Error loading escapes object from {path}: {e}")
    return None

def extract_session_info(session_path):
    session_name = os.path.basename(session_path)
    mouse_name = os.path.basename(os.path.dirname(session_path))
    homings_obj = load_homings_object(session_path)
    escapes_obj = load_escapes_object(session_path)
    num_homings = len(homings_obj.onset_frames) if homings_obj and hasattr(homings_obj, "onset_frames") else 0
    num_escapes = len(escapes_obj.escape_onset_frames) if escapes_obj and hasattr(escapes_obj, "escape_onset_frames") else 0

    # Extract date from session name
    import re
    date_match = re.search(r'(\d{4})[-_](\d{2})[-_](\d{2})', session_name)
    if date_match:
        year, month, day = date_match.groups()
        session_date = pd.Timestamp(year=int(year), month=int(month), day=int(day))
    else:
        session_date = pd.NaT

    return {
        "Mouse Name": mouse_name,
        "Session Name": session_name,
        "Date of Session": session_date,
        "Number of Homings": num_homings,
        "Number of Escapes": num_escapes
    }

# Find all sessions and extract info
session_paths = find_all_sessions()
session_data = [extract_session_info(path) for path in session_paths]
df = pd.DataFrame(session_data)

# Calculate "Days from First Session" for each mouse
df.sort_values(["Mouse Name", "Date of Session"], inplace=True)
df["Days from First Session"] = df.groupby("Mouse Name")["Date of Session"].transform(
    lambda x: (x - x.min()).dt.days if x.notna().any() else np.nan
)

# Select and order columns, including Session Name
df = df[["Mouse Name", "Session Name", "Date of Session", "Days from First Session", "Number of Homings", "Number of Escapes"]]


# Display the DataFrame
print(df.head(10))

   Mouse Name                         Session Name Date of Session  \
21     JAL001       001_seq1_2_2023_03_14T08_11_32      2023-03-14   
20     JAL001    001_mushroom3_2023_03_15T07_43_42      2023-03-15   
22     JAL001       001_seq1_3_2023_03_17T08_38_03      2023-03-17   
23     JAL002    002_mushroom1_2023_04_21T08_24_45      2023-04-21   
27     JAL002    002_Sequence1_2023_04_22T08_38_41      2023-04-22   
26     JAL002       002_seq1_2_2023_04_25T07_32_30      2023-04-25   
24     JAL002    002_mushroom3_2023_04_27T08_30_15      2023-04-27   
28     JAL002  002_Sequence1_3_2023_04_28T08_31_30      2023-04-28   
25     JAL002    002_mushroom4_2023_05_01T08_16_14      2023-05-01   
7      JAL003     003_baseline_2023_08_17T08_16_46      2023-08-17   

    Days from First Session  Number of Homings  Number of Escapes  
21                        0                  0                  4  
20                        1                  0                  0  
22                       

In [6]:
df

,Mouse Name,Session Name,Date of Session,Days from First Session,Number of Homings,Number of Escapes
21,JAL001,001_seq1_2_2023_03_14T08_11_32,2023-03-14,0,0,4
20,JAL001,001_mushroom3_2023_03_15T07_43_42,2023-03-15,1,0,0
22,JAL001,001_seq1_3_2023_03_17T08_38_03,2023-03-17,3,0,2
23,JAL002,002_mushroom1_2023_04_21T08_24_45,2023-04-21,0,0,0
27,JAL002,002_Sequence1_2023_04_22T08_38_41,2023-04-22,1,0,0
26,JAL002,002_seq1_2_2023_04_25T07_32_30,2023-04-25,4,0,3
24,JAL002,002_mushroom3_2023_04_27T08_30_15,2023-04-27,6,0,0
28,JAL002,002_Sequence1_3_2023_04_28T08_31_30,2023-04-28,7,0,8
25,JAL002,002_mushroom4_2023_05_01T08_16_14,2023-05-01,10,0,0
7,JAL003,003_baseline_2023_08_17T08_16_46,2023-08-17,0,5,3


In [7]:
# Remove any rows where Session Name contains "mush" or "tin"
df = df[~df["Session Name"].str.contains("mush|tin", case=False, na=False)]

# Display the filtered DataFrame
df

,Mouse Name,Session Name,Date of Session,Days from First Session,Number of Homings,Number of Escapes
21,JAL001,001_seq1_2_2023_03_14T08_11_32,2023-03-14,0,0,4
22,JAL001,001_seq1_3_2023_03_17T08_38_03,2023-03-17,3,0,2
27,JAL002,002_Sequence1_2023_04_22T08_38_41,2023-04-22,1,0,0
26,JAL002,002_seq1_2_2023_04_25T07_32_30,2023-04-25,4,0,3
28,JAL002,002_Sequence1_3_2023_04_28T08_31_30,2023-04-28,7,0,8
7,JAL003,003_baseline_2023_08_17T08_16_46,2023-08-17,0,5,3
10,JAL003,003_flip_2023_08_22T08_04_56,2023-08-22,5,68,7
11,JAL003,003_flip_rotated_2023_08_25T09_42_06,2023-08-25,8,55,5
9,JAL003,003_flip_1Sept_2023_09_01T16_23_39,2023-09-01,15,109,6
8,JAL003,003_flippuff2_2023_09_04T13_38_31,2023-09-04,18,89,13
